# Unified Dynamic Reactor Model

This is the code as far as my understanding at this moment reaches.
Seperated into concrete sections

- preamble
- kinetic parameters
- and so on

In [ ]:
from dataclasses import dataclass
from typing import Callable, Mapping
import numpy as np
from scipy.integrate import solve_ivp

define class like
e.g.
pbr = simulation(PBR, catalyst (= what kinetics), dataset) 

dataset class:
    dict(length=L,velocity=V,temperature=T,pressure=P,concentration=C,...)    -etc


for key in dataset[key]:
    if catalyst.range[key] is None or not (catalyst.range[key][0] <= dataset[key] <= catalyst.range[key][1]):
        raise OutOfRangeError



    @dataclass
    class Dataset:
        length: float
        velocity: float
        temperature: float
        pressure: float
        concentration: Mapping[str, float]


    @dataclass
    class Catalyst:
        species: tuple[str, ...]
        rate_function: Callable[
            [Mapping[str, float], Dataset],
            Mapping[str, float],
        ]
        ranges: Mapping[str, tuple[float, float] | None] = None

        def validate(self, dataset):
            for name, limits in (self.ranges or {}).items():
                value = getattr(dataset, name)

                if limits is not None and not limits[0] <= value <= limits[1]:
                    raise ValueError(
                        f"{name}={value} is outside the range {limits}"
                    )

        def rates(self, concentration, dataset):
            rates = self.rate_function(concentration, dataset)
            return np.array([
                rates.get(species, 0.0)
                for species in self.species
            ])


    class PBR:
        def __init__(self, catalyst, dataset, cells=50):
            catalyst.validate(dataset)

            self.catalyst = catalyst
            self.dataset = dataset
            self.cells = cells
            self.species = catalyst.species

        def run(self, t, driver=None, solver="BDF"):
            if driver is None:
                driver = lambda time, dataset: dataset.concentration

            n_species = len(self.species)
            dx = self.dataset.length / self.cells
            velocity = self.dataset.velocity

            initial = np.tile(
                [self.dataset.concentration[s] for s in self.species],
                self.cells,
            )

            def rhs(time, state):
                concentration = state.reshape(self.cells, n_species)
                derivative = np.zeros_like(concentration)

                inlet = np.array([
                    driver(time, self.dataset)[s]
                    for s in self.species
                ])

                for cell in range(self.cells):
                    upstream = inlet if cell == 0 else concentration[cell - 1]

                    local_rates = self.catalyst.rates(
                        dict(zip(self.species, concentration[cell])),
                        self.dataset,
                    )

                    derivative[cell] = (
                        -velocity / dx
                        * (concentration[cell] - upstream)
                        + local_rates
                    )

                return derivative.ravel()

            result = solve_ivp(
                rhs,
                (t[0], t[-1]),
                initial,
                t_eval=t,
                method=solver,
            )

            result.concentration = result.y.reshape(
                self.cells,
                n_species,
                len(t),
            )

            return result


    class Simulation:
        def __init__(self, model, catalyst, dataset, **model_options):
            self.model = model(
                catalyst=catalyst,
                dataset=dataset,
                **model_options,
            )

        def run(self, t, **options):
            return self.model.run(t, **options)


    simulation = Simulation
@dataclass
class Dataset:
    length: float
    velocity: float
    temperature: float
    pressure: float
    concentration: Mapping[str, float]

    def __getitem__(self, key):
        return getattr(self, key)

    def items(self):
        return {
            "length": self.length,
            "velocity": self.velocity,
            "temperature": self.temperature,
            "pressure": self.pressure,
            "concentration": self.concentration,
        }.items()


@dataclass
class Catalyst:
    species: tuple[str, ...]
    ranges: Mapping[str, tuple[float, float]]
    rate_function: Callable[[Mapping[str, float], Dataset], Mapping[str, float]]

    def validate(self, dataset: Dataset):
        for key, limits in self.ranges.items():
            value = dataset[key]

            if limits is None:
                raise OutOfRangeError(f"No valid range defined for {key}")

            if not limits[0] <= value <= limits[1]:
                raise OutOfRangeError(
                    f"{key}={value} is outside the range {limits}"
                )

    def rates(self, concentration, dataset):
        rates = self.rate_function(concentration, dataset)
        return np.array([rates.get(species, 0.0) for species in self.species])


class PBR:
    def __init__(self, catalyst, dataset, cells=50):
        self.catalyst = catalyst
        self.dataset = dataset
        self.cells = cells
        self.species = catalyst.species

        catalyst.validate(dataset)

    def rhs(self, time, state, inlet):
        concentration = state.reshape(self.cells, len(self.species))
        derivative = np.zeros_like(concentration)

        dx = self.dataset.length / self.cells
        velocity = self.dataset.velocity

        for cell in range(self.cells):
            upstream = inlet if cell == 0 else concentration[cell - 1]
            local_rate = self.catalyst.rates(
                dict(zip(self.species, concentration[cell])),
                self.dataset,
            )

            derivative[cell] = (
                -velocity / dx * (concentration[cell] - upstream)
                + local_rate
            )

        return derivative.ravel()

    def run(self, t, driver=None, solver="BDF"):
        if driver is None:
            driver = pulse

        initial = np.tile(
            [self.dataset.concentration[s] for s in self.species],
            self.cells,
        )

        def equation(time, state):
            inlet = driver(time, self.dataset)
            inlet = np.array([inlet[s] for s in self.species])
            return self.rhs(time, state, inlet)

        result = solve_ivp(
            equation,
            (t[0], t[-1]),
            initial,
            t_eval=t,
            method=solver,
        )

        result.concentration = result.y.reshape(
            len(self.species),
            self.cells,
            len(t),
        )

        return result


class Simulation:
    def __init__(self, model, catalyst, dataset, **model_options):
        self.reactor = model(
            catalyst=catalyst,
            dataset=dataset,
            **model_options,
        )

    def run(self, t, driver=pulse, solver="BDF"):
        return self.reactor.run(t, driver=driver, solver=solver)


simulation = Simulation


def pulse(time, dataset):
    """Constant inlet concentration."""
    return dict(dataset.concentration)

